# 🚀 RasoSynthTune — 1-Click Cloud Fine-Tuning (Unsloth + LoRA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sourabh-Kumar04/RasoSynth_CUTC/blob/master/finetuning/RasoSynthTune_FineTuning_Colab.ipynb)

Train state-of-the-art LLMs (**Llama-3.1-8B, Mistral-7B, Qwen-2.5, Sarvam-2B**) using datasets generated by **RasoSynthTune**.

- **Engine**: Unsloth FastLanguageModel (2x-5x faster, 70% less VRAM)
- **Hardware**: Free Google Colab T4 GPU or A100/L4
- **Method**: QLoRA (4-bit quantized Low-Rank Adaptation)

### 1. Verify GPU & Install Dependencies

In [ ]:
# Verify GPU allocation
!nvidia-smi

# Install Unsloth and fine-tuning suite
!pip install --no-deps unsloth "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets transformers

### 2. Configuration & Dataset Input
Enter your dataset JSONL URL or file path generated by RasoSynthTune.

In [ ]:
import os

# Set your training parameters
BASE_MODEL = "unsloth/llama-3-8b-Instruct"  # Options: unsloth/mistral-7b-v0.3, Qwen/Qwen2.5-7B-Instruct, sarvamai/sarvam-2b-v0.5
DATASET_URL_OR_PATH = ""  # Paste the download URL from RasoSynthTune or path to uploaded JSONL
OUTPUT_MODEL_NAME = "rasosynthtune-finetuned-model"
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

### 3. Load Model & Tokenizer with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model and LoRA adapters initialized successfully!")

### 4. Load & Format Dataset

In [ ]:
import json
import urllib.request
from datasets import Dataset

# Download dataset if URL is provided
dataset_file = "training_data.jsonl"
if DATASET_URL_OR_PATH.startswith("http"):
    print(f"Downloading dataset from {DATASET_URL_OR_PATH}...")
    urllib.request.urlretrieve(DATASET_URL_OR_PATH, dataset_file)
elif os.path.exists(DATASET_URL_OR_PATH):
    dataset_file = DATASET_URL_OR_PATH
else:
    # Sample fallback for testing
    with open(dataset_file, "w") as f:
        f.write(json.dumps({"instruction": "What is RasoSynthTune?", "input": "", "output": "RasoSynthTune is an autonomous dataset generation and fine-tuning platform with cryptographic DBOM verification."}) + "\n")

# Read and format samples
samples = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{response}"""

def formatting_prompts_func(batch):
    instructions = batch.get("instruction", [])
    inputs = batch.get("input", ["" for _ in instructions])
    responses = batch.get("output", batch.get("response", ["" for _ in instructions]))
    texts = []
    for inst, inp, resp in zip(instructions, inputs, responses):
        text = ALPACA_PROMPT.format(instruction=inst, input=inp, response=resp) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(samples)
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} training samples.")

### 5. Execute SFT Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training completed!")

### 6. Test Model Inference

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [ALPACA_PROMPT.format(
        instruction="Explain how cryptographic DBOM guarantees dataset provenance.",
        input="",
        response=""
    )],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

### 7. Save LoRA Adapters or Push to Hugging Face Hub

In [ ]:
# Save LoRA adapter locally
model.save_pretrained(OUTPUT_MODEL_NAME)
tokenizer.save_pretrained(OUTPUT_MODEL_NAME)
print(f"Saved LoRA adapter to {OUTPUT_MODEL_NAME}")

# Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")
# model.push_to_hub(f"username/{OUTPUT_MODEL_NAME}")
# tokenizer.push_to_hub(f"username/{OUTPUT_MODEL_NAME}")